# m4 follow-up on Colab: the same models trained for 10 epochs

[Open in Colab](https://colab.research.google.com/github/jbouniol/nlp-project/blob/main/colab/run_m4_10epochs.ipynb)

**Why.** In the first m4 runs (`run_m4.ipynb`, 3 epochs), every model was still improving at its last epoch:
validation macro-F1 and validation loss were still getting better, mostly on the rare clause types. This
follow-up trains the two chosen configurations (full fine-tuning at lr 5e-5, LoRA r = 8 on q, v at lr 1e-3)
for 10 epochs; the best epoch is still chosen on validation.

**Rule, fixed before the runs.** For each method, the 10-epoch seed-0 run is compared with the 3-epoch seed-0
run on validation macro-F1 only. If it is better, its test split is predicted once and seeds 1 and 2 are
trained; otherwise the method stops there. The decision is saved to `MyDrive/ledgar/choice_epochs.json`.

**How to use.** *Runtime → Change runtime type → GPU (A100 if possible)*, then *Runtime → Run all*. If the
session is cut, *Run all* again: finished runs are skipped. About 1 h 15 on an A100 (about 4 h on a T4).
**Afterwards**: send `MyDrive/ledgar/results/` and `MyDrive/ledgar/choice_epochs.json`.

## Setup

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ledgar"
RES = f"{DRIVE}/results"
COMMON = f"--out-dir {RES} --checkpoint-dir {DRIVE}/checkpoints --models-dir {DRIVE}/models"
EPOCHS = 10
CONFIGS = {"bert": "--model bert --lr-full 5e-5", "lora": "--model lora --lr-lora 1e-3"}
THREE_EPOCH_RUNS = {"bert": "bert_lr5e-5_seed0", "lora": "lora_lr1e-3_seed0"}

In [ ]:
# The repository at the head of main; Colab's own torch is kept (its CUDA build matches the GPU)
!test -d /content/nlp-project || git clone -q -b main https://github.com/jbouniol/nlp-project.git /content/nlp-project
%cd /content/nlp-project
!git pull -q && git log --oneline -1
!pip install -q transformers==5.19.0 datasets==5.1.0 huggingface-hub==1.33.0 accelerate==1.15.0 peft==0.21.2 scikit-learn==1.9.1
!pip install -q -e . --no-deps

In [ ]:
import json
import os

import torch

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"


def done(name, final=False):
    # a search run is done when its metrics exist; a final run when its test predictions exist
    return os.path.exists(f"{RES}/{name}/pred_test.npz" if final else f"{RES}/search/{name}/metrics.json")


def val_macro_f1(name):
    for path in (f"{RES}/search/{name}/metrics.json", f"{RES}/{name}/metrics.json"):
        if os.path.exists(path):
            return json.load(open(path))["val"]["macro_f1"]
    raise FileNotFoundError(name)

## Step 1: seed 0 for 10 epochs (validation only)

In [ ]:
for method, flags in CONFIGS.items():
    name = f"{THREE_EPOCH_RUNS[method].removesuffix('_seed0')}_ep{EPOCHS}_seed0"
    if not done(name) and not done(name, final=True):
        !python scripts/train.py {flags} --epochs {EPOCHS} --search --seed 0 --name {name} {COMMON}

## Decision on validation: is 10 epochs better than 3?

In [ ]:
choice_path = f"{DRIVE}/choice_epochs.json"
if os.path.exists(choice_path):
    choice = json.load(open(choice_path))
else:
    choice = {}
    for method in CONFIGS:
        three = THREE_EPOCH_RUNS[method]
        ten = f"{three.removesuffix('_seed0')}_ep{EPOCHS}_seed0"
        scores = {"3 epochs": val_macro_f1(three), f"{EPOCHS} epochs": val_macro_f1(ten)}
        choice[method] = {"val_macro_f1": scores, "keep_10_epochs": scores[f"{EPOCHS} epochs"] > scores["3 epochs"]}
    json.dump(choice, open(choice_path, "w"), indent=2)
print(json.dumps(choice, indent=2))

## Step 2: for each method that improved, test predictions of seed 0 (once), then seeds 1 and 2

In [ ]:
for method, flags in CONFIGS.items():
    if not choice[method]["keep_10_epochs"]:
        print(method, ": 3 epochs stay the reference, nothing more to run")
        continue
    stem = f"{THREE_EPOCH_RUNS[method].removesuffix('_seed0')}_ep{EPOCHS}"
    if not done(f"{stem}_seed0", final=True):
        models = f"--models-dir {DRIVE}/models" if method == "bert" else ""
        !python scripts/predict_test.py --name {stem}_seed0 --out-dir {RES} {models}
    for seed in (1, 2):
        if not done(f"{stem}_seed{seed}", final=True):
            !python scripts/train.py {flags} --epochs {EPOCHS} --seed {seed} --name {stem}_seed{seed} {COMMON}

## Summary of the 10-epoch runs on Drive

In [ ]:
import glob

for path in sorted(glob.glob(f"{RES}/**/*_ep{EPOCHS}_seed*/metrics.json", recursive=True)):
    m = json.load(open(path))
    test = m.get("test", {}).get("macro_f1")
    print(f"{path.split('/results/')[1]:<45} val {m['val']['macro_f1']:.4f}" + (f" | test {test:.4f}" if test else ""))